In [4]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

data_path = "phone_dataset"
batch_size = 16
lr = 1e-4

original_epochs = 10
distill_epochs = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using:", device)

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_set = ImageFolder(os.path.join(data_path, "train"), transform=train_tf)
val_set = ImageFolder(os.path.join(data_path, "val"), transform=test_tf)
test_set = ImageFolder(os.path.join(data_path, "test"), transform=test_tf)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
train_loader2 = DataLoader(train_set, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

print(train_set.class_to_idx)

def test_acc(model, loader, dev):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(dev)
            y = y.to(dev)
            out = model(x)
            pred = torch.argmax(out, dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total * 100

def get_size(file):
    return os.path.getsize(file) / 1024 / 1024

def get_latency(model):
    m = copy.deepcopy(model).cpu()
    m.eval()
    x = torch.randn(1, 3, 224, 224)

    with torch.no_grad():
        for _ in range(10):
            m(x)

        t1 = time.time()
        for _ in range(100):
            m(x)
        t2 = time.time()

    return (t2 - t1) / 100 * 1000

def get_ram(model):
    total = 0
    for p in model.parameters():
        total += p.numel() * p.element_size()
    return total / 1024 / 1024

def get_params(model):
    total = 0
    for p in model.parameters():
        total += p.numel()
    return total

print("\ntraining original model")

original_model = torchvision.models.mobilenet_v2(weights="DEFAULT")

for p in original_model.features.parameters():
    p.requires_grad = False

original_model.classifier[1] = nn.Linear(
    original_model.classifier[1].in_features, 2
)

original_model = original_model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(original_model.parameters(), lr=lr)

best_val = 0
best_original = None

for epoch in range(original_epochs):
    original_model.train()

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        out = original_model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()

    tr = test_acc(original_model, train_loader2, device)
    va = test_acc(original_model, val_loader, device)
    te = test_acc(original_model, test_loader, device)

    print("epoch", epoch + 1,
          "train", round(tr, 2), "%",
          "val", round(va, 2), "%",
          "test", round(te, 2), "%")

    if va > best_val:
        best_val = va
        best_original = copy.deepcopy(original_model)

original_model = best_original
torch.save(original_model.state_dict(), "original_model.pth")

original_train = test_acc(original_model, train_loader2, device)
original_val = test_acc(original_model, val_loader, device)
original_test = test_acc(original_model, test_loader, device)

original_flash = get_size("original_model.pth")
original_ram = get_ram(original_model)
original_latency = get_latency(original_model)
original_params = get_params(original_model)

print("\ntraining distilled model")

distilled_model = torchvision.models.MobileNetV2(
    num_classes=2,
    width_mult=0.35
).to(device)

optimizer2 = torch.optim.Adam(distilled_model.parameters(), lr=lr)

T = 4
alpha = 0.7

best_val = 0
best_distilled = None

original_model.eval()

for epoch in range(distill_epochs):
    distilled_model.train()

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        with torch.no_grad():
            teacher_out = original_model(x)

        out = distilled_model(x)

        loss1 = loss_fn(out, y)
        loss2 = F.kl_div(
            F.log_softmax(out / T, dim=1),
            F.softmax(teacher_out / T, dim=1),
            reduction="batchmean"
        ) * (T * T)

        loss = alpha * loss2 + (1 - alpha) * loss1

        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()

    tr = test_acc(distilled_model, train_loader2, device)
    va = test_acc(distilled_model, val_loader, device)
    te = test_acc(distilled_model, test_loader, device)

    print("epoch", epoch + 1,
          "train", round(tr, 2), "%",
          "val", round(va, 2), "%",
          "test", round(te, 2), "%")

    if va > best_val:
        best_val = va
        best_distilled = copy.deepcopy(distilled_model)

distilled_model = best_distilled
torch.save(distilled_model.state_dict(), "distilled_model.pth")

distilled_train = test_acc(distilled_model, train_loader2, device)
distilled_val = test_acc(distilled_model, val_loader, device)
distilled_test = test_acc(distilled_model, test_loader, device)

distilled_flash = get_size("distilled_model.pth")
distilled_ram = get_ram(distilled_model)
distilled_latency = get_latency(distilled_model)
distilled_params = get_params(distilled_model)

print("\nquantizing model")

quant_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(distilled_model).cpu(),
    {nn.Linear},
    dtype=torch.qint8
)

torch.save(quant_model.state_dict(), "quant_model.pth")

quant_train = test_acc(quant_model, train_loader2, "cpu")
quant_val = test_acc(quant_model, val_loader, "cpu")
quant_test = test_acc(quant_model, test_loader, "cpu")

quant_flash = get_size("quant_model.pth")
quant_ram = get_ram(quant_model)
quant_latency = get_latency(quant_model)
quant_params = get_params(quant_model)

print("\n===== final result =====")
print("            original   distilled   quantized")
print("--------------------------------------------")

print(f"train acc   {original_train:6.2f}%    {distilled_train:6.2f}%    {quant_train:6.2f}%")
print(f"val acc     {original_val:6.2f}%    {distilled_val:6.2f}%    {quant_val:6.2f}%")
print(f"test acc    {original_test:6.2f}%    {distilled_test:6.2f}%    {quant_test:6.2f}%")

print()
print(f"flash MB    {original_flash:6.2f}     {distilled_flash:6.2f}     {quant_flash:6.2f}")
print(f"RAM MB      {original_ram:6.2f}     {distilled_ram:6.2f}     {quant_ram:6.2f}")

print()
print(f"params      {original_params:<10} {distilled_params:<10} {quant_params:<10}")

print()
print(f"latency ms  {original_latency:6.2f}     {distilled_latency:6.2f}     {quant_latency:6.2f}")

print()
print(f"compression     -        {original_flash/distilled_flash:5.2f}x     {original_flash/quant_flash:5.2f}x")
print(f"speedup         -        {original_latency/distilled_latency:5.2f}x     {original_latency/quant_latency:5.2f}x")

using: cuda
{'non_phone': 0, 'phone': 1}

training original model
epoch 1 train 78.71 % val 72.73 % test 79.86 %
epoch 2 train 82.76 % val 81.12 % test 85.42 %
epoch 3 train 90.25 % val 87.41 % test 89.58 %
epoch 4 train 89.96 % val 90.21 % test 91.67 %
epoch 5 train 91.3 % val 92.31 % test 90.97 %
epoch 6 train 93.25 % val 93.01 % test 93.06 %
epoch 7 train 92.65 % val 95.8 % test 93.75 %
epoch 8 train 92.2 % val 95.1 % test 94.44 %
epoch 9 train 95.65 % val 95.1 % test 94.44 %
epoch 10 train 93.55 % val 95.8 % test 95.14 %

training distilled model
epoch 1 train 46.78 % val 46.85 % test 46.53 %
epoch 2 train 68.82 % val 65.73 % test 64.58 %
epoch 3 train 73.76 % val 74.83 % test 79.17 %
epoch 4 train 76.91 % val 79.02 % test 84.72 %
epoch 5 train 80.81 % val 83.22 % test 88.19 %
epoch 6 train 88.16 % val 88.11 % test 90.97 %
epoch 7 train 87.11 % val 90.91 % test 91.67 %
epoch 8 train 88.61 % val 87.41 % test 94.44 %
epoch 9 train 88.76 % val 88.81 % test 92.36 %
epoch 10 train 88.61